In [ ]:
                               #######             
                              #                    
        ############ -#####+.##     ######         |	DataDrivenConstruction.io
        ##+########## .-##+ #-   .+########        |	
        ####     ####.     ..    #      ###        |	Dive into the future of flexibility in processing data 
        ####     .#+#.     #    .# .               |    without using BIM tools!
        ####     -#+#.    #.     .#.               |	
        ####      #+#.   ##-                       |	
        ####     ####        #.  #     -####       |	
        ############  .###+ #. ############        |
        #########.   +#### #+.   ########          |	If you have any questions, concerns or special requests,
                           #+                      |	please contact us at info@datadrivenconstruction.io	
                          #                        
                         #  

#------------------------------------Input data-----------------------------------

import os
import sys
import subprocess
from pathlib import Path

# DDC converters and ddc-tools from this repository (see DDC_WINDOWS_Converters/README.md and ddc-tools/README.md)
REPO = Path(os.environ.get("DDC_REPO", r"C:\DDC\cad2data-Revit-IFC-DWG"))
sys.path.insert(0, str(REPO / "ddc-tools" / "src"))   # or install once:  pip install "<repo>\ddc-tools"
from ddc_tools import table

path_ddc = str(REPO / "DDC_WINDOWS_Converters")
rvt_cli = os.path.join(path_ddc, 'DDC_RVT_Converter', 'cli', 'RVTconverter.exe')
ifc_dir = os.path.join(path_ddc, 'DDC_IFC_to_Excel')
ifc_python = os.path.join(ifc_dir, 'runtime', 'python.exe')
ifc_script = os.path.join(ifc_dir, 'sources', 'ifc-excel', 'converter.py')
# Path address of the folder where Revit|IFC files are located
path = os.environ.get("DDC_BATCH", r'C:\IFCprojects')
# Converter results go to <path>\converted\<name>_rvt and <path>\converted\<name>_ifc,
# the flat tables (old layout, sheet Project_Data) to <path>\converted\<name>_rvt.xlsx and <name>_ifc.xlsx

#-----------------------------------------------------------------------------------------

def convert_and_wait(file_path, extension):
    stem = os.path.splitext(os.path.basename(file_path))[0]
    out = os.path.join(path, 'converted', f'{stem}_{extension}')
    if os.path.exists(out):
        print(f'Skipped conversion (result folder exists): {out}')
    else:
        if extension == 'rvt':
            cmd = [rvt_cli, file_path, '-o', out, '--properties', '--ifc']
        else:
            cmd = [ifc_python, ifc_script, file_path, '--output', out]
        subprocess.run(cmd, check=False)   # waits until the converter has finished
    # flat table, the same as:  ddc table <out> --xlsx <out>.xlsx
    xlsx = out + '.xlsx'
    table.write_xlsx(table.build_table(out), xlsx)
    print(f'Conversion done for project {stem}: {out} -> {xlsx}')

# Conversion process from RVT and IFC
os.makedirs(os.path.join(path, 'converted'), exist_ok=True)
for file in os.listdir(path):
    full_path = os.path.join(path, file)
    if file.lower().endswith('.ifc'):
        convert_and_wait(full_path, 'ifc')
    elif file.lower().endswith('.rvt'):
        convert_and_wait(full_path, 'rvt')
